In [ ]:
# Célula 1: Importar bibliotecas
import sys
sys.path.append('.')  # Adiciona o diretório atual ao path

from database import engine
from sqlalchemy import text
import pandas as pd

In [ ]:
# Quais tabelas existem no banco?
with engine.connect() as conn:
    resultado = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name
    """))
    
    tabelas = [row[0] for row in resultado]
    print("📊 TABELAS DO BANCO:")
    print("=" * 40)
    for i, tabela in enumerate(tabelas, 1):
        print(f"{i:2}. {tabela}")

In [21]:
def analisar_tabela(nome_tabela):
    """Função para analisar estrutura e dados de uma tabela"""
    print("\n" + "=" * 60)
    print(f"📋 ANALISANDO TABELA: {nome_tabela.upper()}")
    print("=" * 60)
    
    with engine.connect() as conn:
        # 1. Estrutura da tabela (colunas)
        print("\n📌 ESTRUTURA:")
        estrutura = conn.execute(text(f"""
            SELECT column_name, data_type, is_nullable
            FROM information_schema.columns
            WHERE table_name = '{nome_tabela}'
            ORDER BY ordinal_position
        """))
        
        for col in estrutura:
            print(f"   • {col[0]}: {col[1]} (pode ser nulo? {col[2]})")
        
        # 2. Quantidade de registros
        count = conn.execute(text(f"SELECT COUNT(*) FROM {nome_tabela}")).scalar()
        print(f"\n📌 QUANTIDADE: {count} registros")
        
        # 3. Amostra dos dados (primeiras 3 linhas)
        if count > 0:
            print(f"\n📌 AMOSTRA (primeiras 3 linhas):")
            df = pd.read_sql(f"SELECT * FROM {nome_tabela} LIMIT 3", conn)
            print(df.to_string())
        else:
            print(f"\n📌 AMOSTRA: Tabela vazia")
        
        print("\n" + "-" * 60)

# Analisar cada tabela
for tabela in tabelas:
    analisar_tabela(tabela)


📋 ANALISANDO TABELA: AUDITORIA_PROCESSOS

📌 ESTRUTURA:
   • id: integer (pode ser nulo? NO)
   • auditoria_id: integer (pode ser nulo? YES)
   • processo_id: integer (pode ser nulo? YES)
   • motivo_selecao: text (pode ser nulo? YES)
   • status_avaliacao: character varying (pode ser nulo? YES)
   • created_at: timestamp without time zone (pode ser nulo? YES)
   • updated_at: timestamp without time zone (pode ser nulo? YES)

📌 QUANTIDADE: 19 registros

📌 AMOSTRA (primeiras 3 linhas):
   id  auditoria_id  processo_id                                          motivo_selecao status_avaliacao                 created_at                 updated_at
0  19             1            3  Migração do processo da planilha para o banco de dados         Pendente 2026-03-19 20:24:01.308055 2026-03-23 13:27:58.270405
1   6             1           24  Migração do processo da planilha para o banco de dados         Pendente 2026-03-18 17:56:17.091836 2026-03-23 13:28:22.372035
2  31             1           

In [23]:
def ver_tabela(nome_tabela, linhas=10):
    """Mostra uma tabela do banco de dados"""
    with engine.connect() as conn:
        # Primeiro, contar quantas linhas
        count = conn.execute(text(f"SELECT COUNT(*) FROM {nome_tabela}")).scalar()
        
        print(f"\n{'='*60}")
        print(f"📊 TABELA: {nome_tabela.upper()}")
        print(f"📈 Total de registros: {count}")
        print(f"{'='*60}\n")
        
        # Mostrar as primeiras N linhas
        df = pd.read_sql(f"SELECT * FROM {nome_tabela} LIMIT {linhas}", conn)
        print(df.to_string(index=False))
        
        return display(df)

# Exemplo: ver tabela processos
df_processos = ver_tabela('processos', 10)


📊 TABELA: PROCESSOS
📈 Total de registros: 19

 id                             area codigo_processo                                                                                                                             nome_processo                                                                                         objetivo                             executor                                                                                                                                                                                                                                                                                                                                                                                                                    descricao                                                                                                  etapa_ini                                                                                                             

,id,area,codigo_processo,nome_processo,objetivo,executor,descricao,etapa_ini,etapa_fim,produto,created_at,id_area,status,url_diagrama,link_diagrama,aprovacao,relatorio_gerencial_gerado,data_relatorio_gerencial
0,25,Gerência de Gente e Gestão - GGG,1.5,EMISSÃO DE GUIA NO FGTS DIGITAL (FGTS + CRÉDI...,EMITIR A GUIA DIGITAL (PDF) E FÍSICA DE FGTS M...,TAMIRES (OPERACIONAL),EMISSÃO DA GUIA DE FGTS MENSAL,ACESSAR O FGTS DIGITAL,GUIA FÍSICA DISPONÍVEL PARA LANÇAMENTO NO FINA...,GUIA FÍSICA DE FGTS MENSAL E ARQUIVOS (1º GUIA...,2026-03-04 16:15:14.026809+00:00,1,Ativo,None,None,Em Aprovação,False,None
1,20,Gerência de Gente e Gestão - GGG,1.10,IDENTIFICAÇÃO DA INADIMPLÊNCIA DO FGTS - (PAR...,REGULARIZAÇÃO DA SITUAÇÃO DA DIVIDA FISCAL COM...,RODRIGO LAVINAS OPERACIONAL E GESTÃO,É QUANDO A EMPRESA DEIXA DE RECOLHER O DIREITO...,INICIA NO CONTAS A PAGAR COM O NÃO PAGAMENTO D...,ENVIO DO RELAÓRIO DE CONTAS A PAGAR POR E-MAIL...,EMISSÃO DE RELATÓRIO NO CONTAS A PAGAR,2026-03-04 16:15:14.026809+00:00,1,Ativo,None,None,Em Aprovação,False,None
2,21,Gerência de Gente e Gestão - GGG,1.15,ENVIO DA COMPOSIÇÃO DE PAGAMENTO FGTS - (PARCE...,COMPOSIÇÃO DOS FUNCIONÁRIOS PERTENCENTES A GUI...,RODRIGO LAVINAS OPERACIONAL E GESTÃO,PARA CONFECÇÃO DA GUIA DEVE-SE ATRIBUIR VALOR ...,ACESSO AO SEFIP (ANTIGO) OU ACESSO FGTS DIGITA...,ACESSO AO SEFIP OU FGTS PARA ENVIO DO ARQUIVO ...,GERAÇÃO DE ARQUIVO GUIA MENSAL FGTS TXT OU PDF...,2026-03-04 16:15:14.026809+00:00,1,Ativo,None,None,Em Aprovação,False,None
3,22,Gerência de Gente e Gestão - GGG,1.13,SOLICITAÇÃO DE PARCELAMENTO E CONTRATO CONFIS...,FORMALIZAÇÃO DO CONTRATO E CONFISSÃO DE DIVIDA...,RODRIGO LAVINAS OPERACIONAL E GESTÃO,EM 2013 FOI CONCRETIZADO MANUALMENTE O PEDIDO ...,ACESSO AO CONECTIVIDADE SOCIAL,RECEBIMENTO DO CONTRATO DE PARCELAMENTO (SEFIP...,"ENVIO DO ARQUIVO TXT (ANTIGO) E PDF (ATUAL), ...",2026-03-04 16:15:14.026809+00:00,1,Ativo,None,None,Em Aprovação,False,None
4,24,Gerência de Gente e Gestão - GGG,1.17,PAGAMENTO DE VALOR REMANESCENTE DE PARCELAMENT...,MANTER A CRF (CERTIDÃO DE REGULARIDADE DO FGTS...,RODRIGO LAVINAS,É UM DESDOBRAMENTO DO PARCELAMENTO ANTIGO REFE...,APRESENTAÇÃO DO CONTEXTO E FUNDAMENTO DA COBRANÇA,LANÇAMENTO TOTVS E ENVIO AO FINANCEIRO FUSVE,SOLICITAR E RECEBER VIA Whatsapp e E-mail A GU...,2026-03-04 16:15:14.026809+00:00,1,Ativo,None,None,Em Aprovação,False,None
5,26,Gerência de Gente e Gestão - GGG,1.12,AUTORIZAÇÃO PRESIDENTE PARA DAR ENTRADA NO PA...,VALIDAÇÃO FORMAL DO REPRESENTANTE LEGAL DA FUS...,RODRIGO LAVINAS OPERACIONAL E GESTÃO,DELIBERAÇÃO DO PRESIDENTE COM O COMITÊ GESTOR ...,REUNIÃO PRESENCIAL COM A PRESIDÊNCIA,DELIBERAÇÃO COM AUTORIZAÇÃO DO PARCELAMENTO,"APRESENTAÇÃO DO RELATÓRIO, ANALISE DA DIVIDA A...",2026-03-04 16:15:14.026809+00:00,1,Ativo,None,None,Em Aprovação,False,None
6,27,Gerência de Gente e Gestão - GGG,1.9,FGTS 13º SALÁRIO (2º PARCELA),GERAR A INFORMAÇÃO PARA PAGAR A 2º PARCELA DO ...,TODA EQUIPE DE GESTÃO DE PESSOAS,CALCULO DA FOLHA DE PAGAMENTO 2º PARCELA DO 13...,DENTRO DO SISTEMA TOTVS (RH) ACIONAR O BOTÃO C...,SALVA NO ONEDRIVE NA PASTA 13 DEZEMBRO E ANO V...,GERADO TOTVS RH O RECIBO INDIVIDUALIZADO DE 2º...,2026-03-04 16:15:14.026809+00:00,1,Ativo,None,None,Em Aprovação,False,None
7,3,Gerência de Gente e Gestão - GGG,1.1,"GERAR CONTRACHEQUE TOTVS (RH) (SALÁRIO, FÉRIAS...",Garantir GERAR NO SISTEMA TOTVS O PAGAMENTO DE...,TODA EQUIPE DE GESTÃO DE PESSOAS,CALCULO DA FOLHA DE PAGAMENTO MENSAL DO FUNCIO...,JOGAR O GRUPO DE EVENTOS PARA O CONTRACHEQUE N...,"SALVA NO ONDRIVE ""ENVELOPE DE PAGAMENTO FILIAL...",GERAÇÃO DO CONTRACHEQUE,2026-03-04 13:49:37.933225+00:00,1,Ativo,None,None,Em Aprovação,False,None
8,28,Gerência de Gente e Gestão - GGG,1.4,CONFERÊNCIA DE VALORES - EMPRESTIMO CRÉDITO DO...,VALIDAÇÃO DOS VALORES DOEMPRESTIMO CRÉDITO DO ...,TAMIRES (OPERACIONAL),CONFERIR SE AS INFORMAÇÕES GERADAS NO PROCESSO...,ARQUIVO EM EXCEL CREDITO DO TRABALHADOR E ACES...,"SALVA NO ONDRIVE ""CONF EMPRESTIMO MÊS/ANO""","PLANILHA ""CONF EMPRESTIMO M